In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy
from scipy.signal import find_peaks
from lib.readwav import *

# Classificazione di note prodotte da un organo


In [45]:
# Funzione di finestra gaussiana 
def apply_window(rate, cut_note, sigma):
    sigma *= rate
    N = cut_note.shape[0]
    x = np.arange(N)
    center = 0.50 * N
    window = np.exp(-0.5 * ((x - center) / sigma) ** 2)
    return cut_note * window

In [46]:
fundamental_frequencies = [] # Frequenza fondamentale di ogni nota ottenuta come media dall'istogramma
harmonics = [] # Frequenza fondamentale e prime 5 armoniche ottenute come selezione dei picchi
power_spectra_harmonics = [] # Valori di spettro di potenza per frequenza fondamentale e prime 5 armoniche
normalized_power_spectra_harmonics = [] # Valori normalizzati di spettro di potenza per la clusterizzazione

In [49]:
for i in range(1,281):
   # Import di tutti i file audio, rates = 280 frequenze di campionamento (48kHz), note = 280 array di dati (un canale del file audio)
   filename = "./wav_files/note_{}.wav".format(i)
   rate, note = readwav(filename)
   time = np.linspace(0, note.shape[0], note.shape[0], endpoint = False) / rate
   note = note[:,0]

   # # Cut di attack e decay
   first_15 = time[0] + 0.15 * (time[-1] - time[0])
   last_5 = time[0] + 0.95 * (time[-1] - time[0])
   cut = (time >= first_15) & (time <= last_5)
   
   note = note[cut]
   # time = time[cut]

   # Applicazione della finestra gaussiana
   masked_note = apply_window(rate, note, sigma = 0.5)
   
   # Calcolo dello spettro di potenza con finestra gaussiana mobile
   segment_size = int(rate * 0.05) # il numero è in secondi, quindi la variabile si riferisce agli indici (0.05s => 2000 indici)
   hop_size = segment_size // 4 # hop_size essere un quarto della window è arbitrario, ma segment dev'essere multiplo intero di hop

   number_of_segments = (len(note) - segment_size) // hop_size + 1 # questa è la formula giusta per contare gli hop senza sforare!
   # Il punto è che le finestre si overlappano: ad ogni iterazione mi muovo avanti di un hop e controllo la window che comincia lì
   # L'idea di base è che facendo la media tra i vari spettri rimuoviamo il rumore bianco, perché il rumore bianco ha media nulla!

   segments_spectra = []
   for j in range(number_of_segments):
      start = j * hop_size
      end = start + segment_size
      segment = note[start:end]
      segment = apply_window(rate, segment, sigma = 0.1)
      # sigma più grande (largo) non migliora né peggiora, sigma più piccolo rovina la risoluzione
      # quindi teniamo questo sigma fisso come compromesso risoluzione-rumore (vedi anche figure sopra)
      # e la risoluzione la aggiustiamo qui solo sulla segment size
      power_spectrum = scipy.fft.rfft(segment)
      power_spectrum = np.abs(power_spectrum)**2
      segments_spectra.append(power_spectrum)

   averaged_spectrum = np.mean(segments_spectra, axis = 0)
   frequencies = scipy.fft.rfftfreq(segment.shape[0], d=1/rate) # tanto le frequenze fondamentali sono le stesse in ogni segment

   prom = 10**(np.floor(np.log10(averaged_spectrum[0]))+1) # occhio al +1 per prendere un picco molto alto
   previsional_peaks, _ = find_peaks(averaged_spectrum, prominence = prom)
   previsional_frequency = frequencies[previsional_peaks[0]]


   # Seconda iterazione per calcolare lo spettro di frequenza, basandosi sulla frequenza trovata nella prima iterazione
   A = -0.03 # dal fit rudimentale della cella sopra
   B = 0.35
   segment_size = int(rate * (A * np.log(previsional_frequency) + B) ) # line importante!
   hop_size = segment_size // 4
   number_of_segments = (len(note) - segment_size) // hop_size + 1

   segments_spectra = []
   for i in range(number_of_segments):
      start = i * hop_size
      end = start + segment_size
      segment = note[start:end]
      segment = apply_window(rate, segment, sigma = 0.17)
      # sigma più grande (largo) non migliora né peggiora, sigma più piccolo rovina la risoluzione
      # quindi teniamo questo sigma fisso come compromesso risoluzione-rumore (vedi anche figure sopra)
      # e la risoluzione la aggiustiamo qui solo sulla segment size
      power_spectrum = scipy.fft.rfft(segment)
      power_spectrum = np.abs(power_spectrum)**2
      segments_spectra.append(power_spectrum)

   averaged_spectrum = np.mean(segments_spectra, axis = 0)
   frequencies = scipy.fft.rfftfreq(segment.shape[0], d=1/rate)

   prom = 10**(np.floor(np.log10(averaged_spectrum[0]))) # senza il +1 (provare +0.5 o senza floor ??)

   corrected_peaks, _ = find_peaks(averaged_spectrum, prominence = prom)
   if len(corrected_peaks) < 5:
      prom = 10**(np.floor(np.log10(averaged_spectrum[0]))-1)
      corrected_peaks, _ = find_peaks(averaged_spectrum, prominence = prom)
   frequencies_of_peaks = frequencies[corrected_peaks]

   ###################### Forse l'idea della tolerance non è male dopotutto, dobbiamo pensare ad una
   # implementazione più intelligente, nella scelta della tolerance e nella selezione del più alto tra due picchi vicini #################

   tolerance = previsional_frequency / 2
   frequencies_of_peaks = []
   for peak in corrected_peaks:
      aux = frequencies[peak]
      if all(abs(aux - f) > tolerance for f in frequencies_of_peaks):
         frequencies_of_peaks.append(aux)


   # Identificazione della frequenza fondamentale
   peaks_distances = np.diff(frequencies_of_peaks)

   counts, bin_edges = np.histogram(peaks_distances, bins=20) 
   index_max = np.argmax(counts)

   distances_in_bin = peaks_distances[(peaks_distances >= bin_edges[index_max]) & (peaks_distances <= bin_edges[index_max + 1])]
   fundamental = np.mean(distances_in_bin)
   dev = np.std(distances_in_bin)

   fundamental_frequencies.append(fundamental)   

   # Identificazione delle armoniche successive alla fondamentale
   estimated_harmonics = [fundamental * i for i in range(1,7)] # 6 elementi, consideriamo le prime 6 armoniche
   #tolerance_harmonics = 30 # commentato per usare una tolleranza dinamica
   measured_harmonics_frequency = []
   measured_harmonics_power = []
   #considered_harmonics = corrected_peaks[:9] # commentato per considerare più picchi sperimentali possibili
   neighborhood = 5 # nel caso in cui prendiamo l'armonica teorica, come potenza prendiamo il massimo in un intorno di 5 indici

   for estimate in estimated_harmonics:
      index_closest_harmonic = np.argmin(np.abs(estimate - frequencies_of_peaks))
      closest_harmonic = frequencies_of_peaks[index_closest_harmonic]
      
      if abs(estimate - closest_harmonic) < (0.05 * closest_harmonic):
         measured_harmonics_frequency.append(closest_harmonic)
         index_target = np.argmin(np.abs(closest_harmonic - frequencies))
         measured_harmonics_power.append(averaged_spectrum[index_target])
      else:
         measured_harmonics_frequency.append(estimate)
         index_target = np.argmin(np.abs(estimate - frequencies))
         start = max(0, index_target - neighborhood)
         end = min(len(averaged_spectrum), index_target + neighborhood + 1)
         measured_harmonics_power.append(np.max(averaged_spectrum[start:end]))

   harmonics.append(measured_harmonics_frequency)
   power_spectra_harmonics.append(measured_harmonics_power)
   # normalizzazione per clusterizzazione
   normalized_power = measured_harmonics_power / measured_harmonics_power[0]
   normalized_power_spectra_harmonics.append(normalized_power)


In [50]:
# save to file
np.save("fundamental_frequencies.npy", fundamental_frequencies)
np.save("harmonics.npy", harmonics)
np.save("power_spectra_harmonics.npy", power_spectra_harmonics)
np.save("normalized_power_spectra_harmonics.npy", normalized_power_spectra_harmonics)

